# Tutorial: QSAR Descriptor Smoke Test

Audience:
- AMDockVS developers.

Prerequisites:
- Ligand import works.

Learning goals:
- Verify the basic integration with runtime.qsar.compute_descriptors().
- Confirm that descriptors are available through the public API.

## Outline

1. Set up
2. Import a sample ligand
3. Compute descriptors
4. Inspect the public output

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="04_qsar_descriptors_", dir=TMP_ROOT))
LIGAND_FILE = REPO_ROOT / "data" / "docking" / "1iep_ligand.sdf"
assert LIGAND_FILE.exists(), LIGAND_FILE

runtime = AMDockVSRuntime()
runtime.create_project(
    name=f"qsar-descriptors-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for descriptor calculation.",
)

import_job_ids = runtime.loader.load_ligands([LIGAND_FILE], executor_name="process")
import_status = runtime.wait_for_jobs(import_job_ids, timeout_s=180.0, poll_s=0.2)
assert all(row.status == "completed" for row in import_status.values()), import_status
import_status


In [ ]:
descriptor_job_id = runtime.qsar.compute_descriptors(
    batch_size=32,
    executor_name="process",
    depends_on=import_job_ids,
)
descriptor_status = runtime.wait_for_job(descriptor_job_id, poll_s=0.2)
assert descriptor_status.status == "completed", descriptor_status
descriptor_status


In [ ]:
descriptors = runtime.qsar.list_descriptors()
assert descriptors, "Expected descriptor rows"

row = descriptors[0]
assert row.status == "completed", row
assert row.mw is not None, row
assert row.metadata.get("stored_path"), row.metadata

{
    "descriptor_rows": len(descriptors),
    "first_row": {
        "ligand_id": row.ligand_id,
        "mw": row.mw,
        "logp": row.logp,
        "hbd": row.hbd,
        "hba": row.hba,
        "status": row.status,
        "stored_path": row.metadata.get("stored_path"),
    },
}
